# Lab 07 — Why Schema Validation in ML/LLM Pipelines
### Week 2 · Data Engineering for LLM Pipelines

Labs 05–06 built a pipeline that *produces* a clean feature table. This lab asks the
next question: **how do you know it's still clean tomorrow?** Upstream systems drift —
a type flips, a new category sneaks in, a negative value appears — and none of it
raises an error on its own. It just quietly poisons whatever runs next: a transform, a
metric, a training set, a prompt.

You'll simulate three kinds of drift on a Cordwell feature table, then build a
**validation gate** — with **pandera** for bulk DataFrames and **pydantic** for
per-row payloads — that fails fast with an actionable report *before* bad data reaches
an LLM-adjacent stage.

**By the end you will be able to:**
1. Distinguish **structural drift** (columns/types) from **semantic drift**
   (values/ranges/categories), and name how each breaks a downstream LLM component.
2. Write a **pandera `DataFrameSchema`** that enforces dtypes, ranges, and categorical
   sets, and read its failure report.
3. Write a **pydantic** row model for API-boundary / per-record validation.
4. Wrap it all in a reusable **pre-flight gate** that blocks a bad batch and writes a
   triage report.

> **Hints stay light (see Labs 05–06).** Each Part opens with a **Toolbox**; picking
> the right tool is part of the work. Target **18/18**; a red check never halts the
> notebook. **Watch dtypes** — the sharpest trap in this lab is a schema that looks
> right and rejects *clean* data on pandas 3.0.


## Setup — a clean feature table & the `check()` helper

In [ ]:
%pip install -r requirements.txt

In [ ]:
import numpy as np
import pandas as pd
import pandera.pandas as pa          # modern, warning-free import (NOT `import pandera as pa`)
from pydantic import BaseModel, Field, ValidationError
from typing import Literal

print("pandas", pd.__version__, "| pandera", pa.__version__ if hasattr(pa,"__version__") else "?")

def build_cordwell_features(seed=42, n=300):
    """A clean feature table, as if emitted by the Lab 06 pipeline."""
    rng = np.random.default_rng(seed)
    age = rng.integers(16, 80, size=n).astype("int64")
    ltv = np.round(np.clip(rng.lognormal(3.0, 0.7, size=n), 0, 5e4), 2)
    hv_threshold = np.quantile(ltv, 0.85)
    return pd.DataFrame({
        "customer_id":   [f"C{i:05d}" for i in range(n)],
        "country_norm":  rng.choice(["USA","DE","SG","BR"], size=n, p=[.55,.2,.15,.1]),
        "age":           age,
        "ltv_usd":       ltv,
        "is_adult":      age >= 18,
        "is_high_value": ltv >= hv_threshold,
    })

users = build_cordwell_features()
print("shape:", users.shape)
print(users.dtypes)
users.head()

In [ ]:
# ── soft self-check: prints PASS/FAIL, never raises ──────────────────────────
_score = {"pass": 0, "fail": 0}
def check(label, predicate):
    try:
        ok = bool(predicate() if callable(predicate) else predicate); note = ""
    except Exception as e:
        ok, note = False, f"  [error: {type(e).__name__}: {e}]"
    _score["pass" if ok else "fail"] += 1
    print(f"{'✅ PASS' if ok else '❌ FAIL'} — {label}{note}")
def score():
    t = _score["pass"] + _score["fail"]
    print(f"\n{'='*46}\n  {_score['pass']}/{t} checks passing  ({_score['fail']} to go)\n{'='*46}")

def raises(fn, exc=Exception):
    """True iff calling fn() raises `exc` (used to assert a gate fires)."""
    try:
        fn(); return False
    except exc:
        return True

In [ ]:
check("Setup: clean table is 300 x 6", lambda: users.shape == (300, 6))
check("Setup: string columns are 'str' dtype (pandas 3.0), not 'object'",
      lambda: users["customer_id"].dtype == "str" and users["country_norm"].dtype == "str")

---
## Part A — What can go wrong, concretely?

Drift comes in two flavors, and the distinction drives how you catch it:

- **Structural drift** — the *shape* changes: a column's **dtype** flips (age arrives
  as a string), a column disappears, a new one appears. Breaks **transforms** and
  **joins** outright.
- **Semantic drift** — the shape is fine but the *values* go out of policy: an unseen
  **category** (`"U.S.A."` returns after you normalized it away), an out-of-**range**
  number (negative price, age 200), a null where you require a value. Breaks **metrics**
  and **poisons train/eval** silently.

Build a `broken` copy with **three** planted drifts so we have something to catch.

> **🧰 Toolbox for Part A** — `.copy()` · `.astype(str)` · `.loc[rows, col] = [...]` ·
> `Series.isin(...)` · boolean masks.

### A1 — Structural drift: `age` arrives as a string

Simulate an upstream type flip: make the **entire `age` column** string dtype on
`broken`. (In pandas 3.0 you can't splice strings into part of an int column — it
raises. Flip the whole column.)

In [ ]:
broken = users.copy()
# TODO: flip the whole `age` column to string dtype on `broken`.
print("age dtype:", broken["age"].dtype)

In [ ]:
check("A1: age drifted to string dtype", lambda: broken["age"].dtype == "str")
check("A1: users.age untouched (still int64)", lambda: users["age"].dtype == "int64")

### A2 — Semantic drift: out-of-policy country labels

Policy allows exactly `{USA, DE, SG, BR}`. Inject the un-normalized spellings Lab 06
taught you to clean — set rows **10–13** of `country_norm` to
`["U.S.A.", "United States", "usa", "US"]`.

In [ ]:
allowed = ["USA", "DE", "SG", "BR"]
# TODO: set rows 10-13 of broken["country_norm"] to 4 out-of-policy spellings, then
#       count how many rows are now outside `allowed`.
n_invalid = 0
print("out-of-policy country rows:", int(n_invalid))

In [ ]:
check("A2: 4 out-of-policy country rows injected",
      lambda: int((~broken["country_norm"].isin(["USA","DE","SG","BR"])).sum()) == 4)

### A3 — Range drift: negative lifetime value

Money can't be negative. Push rows **50–52** of `ltv_usd` to `[-10.0, -5.0, -1.0]`.

In [ ]:
# TODO: set rows 50-52 of broken["ltv_usd"] negative, then count rows below 0.
n_negative = 0
print("negative ltv rows:", int(n_negative))

In [ ]:
check("A3: 3 negative ltv rows injected", lambda: int((broken["ltv_usd"] < 0).sum()) == 3)

---
## Part B — A pandera schema as the gate

One `DataFrameSchema` encodes the whole contract: dtype + value rules per column.
Validate the clean frame (passes) and the broken frame (fails with a report you can
attach to CI or Slack).

> **🧰 Toolbox for Part B** — `pa.DataFrameSchema({...})` · `pa.Column(dtype, check,
> nullable=, unique=)` · `pa.Check.isin([...])` · `pa.Check.in_range(lo, hi)` ·
> `pa.Check.ge(0)` · `pa.Int64` · `schema.validate(df, lazy=True)` ·
> `pa.errors.SchemaErrors` · `err.failure_cases`.
>
> ⚠️ **The dtype trap.** On pandas 3.0 your string columns are `str`, **not**
> `object`. A column declared `pa.Column(object, ...)` — the spelling in every
> pre-3.0 tutorial — will reject your **clean** data. Declare string columns as
> `pa.Column(str, ...)`.

### B1 — Define the contract

Write `Schema` so it accepts the clean table and encodes every rule:
`customer_id` non-null **and unique**; `country_norm` in the allowed set; `age` an
**integer** in `[0, 120]`; `ltv_usd` a float `≥ 0`; `is_adult` and `is_high_value`
booleans. Nothing nullable.

In [ ]:
allowed = ["USA", "DE", "SG", "BR"]
# TODO: build a DataFrameSchema with one Column per field, correct dtypes + rules.
#       (Remember the dtype trap: string columns are `str`, not `object`.)
Schema = pa.DataFrameSchema({
    "customer_id": pa.Column(str, nullable=False),
})
print("schema columns:", list(Schema.columns))

In [ ]:
check("B1: schema covers all 6 columns",
      lambda: set(Schema.columns) == {"customer_id","country_norm","age","ltv_usd","is_adult","is_high_value"})
check("B1: schema ACCEPTS the clean table (dtype trap avoided)",
      lambda: len(Schema.validate(users, lazy=True)) == 300)

### B2 — Validate clean vs broken; capture the report

Validate `users` (should pass) and `broken` (should fail). Catch
`pa.errors.SchemaErrors` and keep its `.failure_cases` table as `report`.

*Why `lazy=True`?* Without it, validation stops at the **first** failure. Lazy collects
**every** violation into one report — which is what triage needs.

In [ ]:
clean_rows = len(Schema.validate(users, lazy=True))
print("clean rows validated:", clean_rows)

# TODO: validate `broken` under lazy=True, catch SchemaErrors, store err.failure_cases -> report
report = None
print("total failure cases:", 0 if report is None else len(report))
report

In [ ]:
check("B2: clean frame fully validated (300 rows)", lambda: clean_rows == 300)
check("B2: broken frame produced a report of 9 failure cases", lambda: report is not None and len(report) == 9)
check("B2: failures span age, country_norm, ltv_usd",
      lambda: set(report["column"].unique()) == {"age","country_norm","ltv_usd"})

### B3 — Roll it up for CI / Slack

The raw report is per-row. Collapse it to **one line per (column, check)** with a
`failures` count, sorted worst-first — the artifact you'd attach to a failing build.

In [ ]:
# TODO: group report by (column, check), count rows as `failures`, sort worst-first.
summary = pd.DataFrame(columns=["column", "check", "failures"])
summary

In [ ]:
check("B3: summary has one row per (column, check) — 4 rows", lambda: len(summary) == 4)
check("B3: worst offender is country_norm (4 failures)",
      lambda: summary.iloc[0]["column"] == "country_norm" and int(summary.iloc[0]["failures"]) == 4)

---
## Part C — Row-level validation with pydantic

Pandera gates **bulk DataFrames** in ETL. **pydantic** gates **one record at a time** —
the right tool at an API boundary or message-queue consumer, where data arrives as a
dict, not a frame.

> **🧰 Toolbox for Part C** — `class X(BaseModel)` · type hints · `Literal[...]` for
> categorical sets · `Field(ge=, le=)` for ranges · `Model(**row_dict)` ·
> `ValidationError` · `df.iloc[i].to_dict()`.

### C1 — A row contract

Write `CustomerRow` mirroring the schema: `country_norm` restricted to the four codes,
`age` an int in `[0, 120]`, `ltv_usd` a float `≥ 0`. Confirm a clean row validates and
a broken row (row 50 — negative ltv) raises `ValidationError`.

In [ ]:
# TODO: define CustomerRow(BaseModel) mirroring the schema rules, then show a clean row
#       validates and broken.iloc[50] (negative ltv) raises ValidationError.
class CustomerRow(BaseModel):
    customer_id: str
    # ... the other five fields, with the right constraints

good = None
bad_raised = False
print("bad_raised:", bad_raised)

In [ ]:
check("C1: clean row validates through pydantic", lambda: good is not None and good.customer_id == "C00000")
check("C1: row with negative ltv is rejected", lambda: bad_raised is True)

---
## Part D — The pre-flight gate

Wrap validation in one reusable function the pipeline calls **before** it trusts a
batch: return the validated frame on success; on failure, **write a CSV report** for
triage and **raise** a concise error so the pipeline stops.

> **🧰 Toolbox for Part D** — `Path(...).mkdir(parents=True, exist_ok=True)` ·
> `try/except pa.errors.SchemaErrors` · `DataFrame.to_csv(path, index=False)` ·
> `raise RuntimeError(msg)` · f-strings.

### D1 — `validate_or_raise` + fail-fast

Implement it to the contract in the docstring. Then prove it: it **returns** the clean
frame, and it **raises** on `broken` while leaving a CSV behind.

In [ ]:
from pathlib import Path

def validate_or_raise(df, schema, name, out_dir="artifacts/validation"):
    """Validate df against schema. On success return the validated frame.
    On failure: write <name>_schema_failures.csv to out_dir and raise RuntimeError
    with a compact summary of the top issues."""
    # TODO: implement to the contract above (mkdir; validate lazy; on SchemaErrors
    #       write CSV + raise RuntimeError; else return validated frame).
    return schema.validate(df, lazy=True)

clean_out = validate_or_raise(users, Schema, name="users_clean")
gate_blocked = raises(lambda: validate_or_raise(broken, Schema, name="users_broken"), RuntimeError)
report_written = (Path("artifacts/validation") / "users_broken_schema_failures.csv").exists()
print("gate blocked:", gate_blocked, "| report written:", report_written)

In [ ]:
check("D1: gate passes the clean batch (returns 300 rows)", lambda: len(clean_out) == 300)
check("D1: gate raises RuntimeError on the broken batch", lambda: gate_blocked is True)
check("D1: gate wrote a CSV triage report", lambda: report_written is True)
score()

---
## Wrap-up — answer in this Markdown cell

1. **One structural, one semantic.** Name the drift you planted in each category and
   the downstream LLM component it would break (a transform? a metric? a training set?).
2. **Top three failures.** Paste your B3 summary and, for each, say whether you'd fix it
   **at the source** (upstream contract) or with a **transform rule** (clean-and-continue).
3. **Placement.** Where in the Lab 05–06 pipeline does this gate belong, and why *there*?

**Key takeaways**
- Drift is **silent** — it doesn't raise on its own. A schema is how you make it loud.
- **`str`, not `object`.** On pandas 3.0 a schema written for the old string dtype
  rejects clean data. Declare string columns `pa.Column(str, ...)`.
- `lazy=True` collects **every** violation into one report — triage needs the whole
  picture, not just the first failure.
- **pandera for frames, pydantic for records.** ETL batches vs API-boundary payloads.
- A gate **returns-or-raises**: pass the good batch through, block the bad one, and
  leave a report behind.

> ⚠️ **CURRENCY FLAG — pydantic coerces by default.** In lax mode, pydantic v2 will
> turn a stringified `"63"` into `63` and pass it. If you want the row model to *catch*
> a type drift (string where an int belongs), enable strict mode — e.g.
> `model_config = ConfigDict(strict=True)` or `Field(strict=True)`. Pandera caught the
> `age` type flip; a lax pydantic model would not.
